## 1. Imports

In [1]:
import os
import subprocess
import mwxml
import re
import numpy as np
from tqdm import tqdm

## 2. Check if data is available

In [2]:
print("Checking for required data:")
if(not(os.path.exists("../Data/dewiki-latest-pages-articles-multistream-index.txt"))):
    print("Index-File not found. Download:")
    subprocess.call(['sh', '../bin/download-and-unzip-index.sh'])
print("Index-File available at: ../Data/dewiki-latest-pages-articles-multistream-index.txt")

if(not(os.path.exists("../Data/dewiki-latest-pages-articles-multistream.xml"))):
    print("Articles-File not found. Download (this might take a while):")
    subprocess.call(['sh', '../bin/download-and-unzip-data.sh'])
print("Articles-File available at: ../Data/dewiki-latest-pages-articles-multistream.xml")

Checking for required data:
Index-File available at: ../Data/dewiki-latest-pages-articles-multistream-index.txt
Articles-File available at: ../Data/dewiki-latest-pages-articles-multistream.xml


## 3. Load all Wikipedia Articles and extract Text

In [4]:
dump = mwxml.Dump.from_file(open("../Data/dewiki-latest-pages-articles-multistream.xml"))

def dump_to_dataset(dump:mwxml.iteration.dump.Dump, n_samples:int=-1) -> tuple([np.ndarray, np.ndarray]):
    data = np.memmap('../Data/article_text.dat', dtype='object', mode='w+', shape=(n_samples, 1))
    label = np.memmap('../Data/label.dat', dtype=np.int8, mode='w+', shape=(n_samples, 1))
    i = 0
    pbar = tqdm(total= (n_samples if (n_samples>1) else 50000000))
    for page in dump:
        for revisions in page:
            try:
                if("Liste von Autoren" not in revisions.page.title):
                    if(re.search(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", revisions.text)):
                        label[i] = 1
                        data[i] = str(re.sub(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", '',revisions.text))
                    else:
                        label[i] = 0
                        data[i] = str(revisions.text)
                    i += 1
                    pbar.update(1)
            except Exception as e:
                pass
        if(i>=n_samples):
            break
    pbar.close()
    return np.array(data).ravel(), np.array(label)

In [5]:
features, labels = dump_to_dataset(dump, 100000)

100%|██████████| 100000/100000 [00:17<00:00, 5568.46it/s]


## 4. Train Neural Network

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

### 4.1 Tokenize Text of Articles

In [ ]:
tokenizer = Tokenizer(
    num_words=10000,
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)
tokenizer.fit_on_texts(features)

In [ ]:
sequences = tokenizer.texts_to_sequences(features)

### 4.2 Padd Sequences for unified length

In [ ]:
padded_sequences = pad_sequences(sequences, maxlen=10000)

### 4.3 Build and compile Neural Network

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=10000, output_dim=64),
    tf.keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', weighted_metrics=['accuracy'])

### 4.4 Train Neural Network

In [ ]:
model.fit(padded_sequences, labels, epochs=100, batch_size=75, validation_split=0.2)